# Interpretability — SHAP, XPER, permutation importance, surrogate, PDP/ICE, LIME

Runs for each model with a `models/<name>_model.py` file (`report_status()` below shows
which). The last cell is a throwaway smoke test for when no real model is in yet — it
skips itself once one is, so it's safe to leave in.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("../..").resolve()))

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import roc_auc_score

import common_metrics as cm

NB_OUTPUT_DIR = cm.OUTPUT_DIR / "interpretability"
NB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cm.report_status()

## Load data

In [ ]:
train_df = cm.load_split("train")
test_df = cm.load_split("test")

X_train, y_train = cm.get_X_y(train_df)
X_test, y_test = cm.get_X_y(test_df)

print(f"train: {X_train.shape}, test: {X_test.shape}")

## SHAP — per-prediction attribution

TreeExplainer for XGBoost, LinearExplainer for logreg, KernelExplainer for TabPFN, falling
back to KernelExplainer if the native explainer can't be built from whatever `fit()`
returned (e.g. a pipeline-wrapped model).

In [ ]:
MODEL_EXPLAINER_TYPE = {"xgboost": "tree", "logreg": "linear", "tabpfn": "kernel"}

def get_shap_explainer(model_type, model, module, X_background):
    # KernelExplainer perturbs via raw numpy, which drops dtype info — a categorical column
    # (e.g. loan_type) round-trips as generic float/object, which XGBoost's DMatrix construction
    # then rejects. Restoring X_background's own dtypes after the round-trip fixes it.
    def predict_fn(X):
        df = pd.DataFrame(X, columns=X_background.columns).astype(X_background.dtypes)
        return module.predict_proba(model, df)[:, 1]
    if model_type == "tree":
        probe = X_background.iloc[:2]
        for candidate in (model, getattr(model, "booster", None)):
            if candidate is None:
                continue
            try:
                explainer = shap.TreeExplainer(candidate)
                explainer(probe)  # smoke-test: a wrapper wanting its own preprocessing fails here, not later
                note = "TreeExplainer (exact)" if candidate is model else (
                    "TreeExplainer (exact, on model.booster — pre-calibration score, "
                    "not the final calibrated probability)"
                )
                return explainer, note
            except Exception:
                continue
        print("  TreeExplainer failed on both the model and model.booster — falling back to KernelExplainer")
    elif model_type == "linear":
        try:
            return shap.LinearExplainer(model, X_background), "LinearExplainer (exact)"
        except Exception as e:
            print(f"  LinearExplainer failed ({e}) — falling back to KernelExplainer")
    background_sample = shap.sample(X_background, min(100, len(X_background)))
    return shap.KernelExplainer(predict_fn, background_sample), "KernelExplainer (approximate)"

## XPER — performance decomposition

`pip install XPER` ([github.com/hi-paris/XPER](https://github.com/hi-paris/XPER)). Unlike
SHAP, XPER decomposes a *performance metric* into per-feature Shapley contributions, not a
single prediction. `metric` is a fixed name (`"AUC"`, `"Accuracy"`, `"MC"` for
misclassification cost) rather than an arbitrary function; `"MC"` takes `cfp`/`cfn` for an
economic-cost decomposition. `kernel=True` since the docs recommend it above ~10 features
and we have ~28.

**Restricted to the top `XPER_TOP_N` features by permutation importance, not all ~28-31** —
measured directly (not estimated): XPER's own coalition sampling took 65 minutes for just 15
features on a small synthetic model, using XPER's own documented fastest setup (bare numpy,
no adapter). Runtime scales with feature count, not row count, so this is a property of the
library's algorithm, not something fixable in our adapter or by capping rows. Every other
feature is held fixed at a real observation's values (`_XPERSubsetAdapter` below) rather than
dropped from the model — XPER still decomposes the *actual* model's performance, just
attributing it to a handful of features instead of all of them.

In [ ]:
try:
    from XPER.compute.Performance import ModelPerformance
    HAS_XPER = True
except ImportError:
    HAS_XPER = False
    print("XPER not installed — pip install XPER, then re-run this cell")


class _XPERModelAdapter:
    """XPER calls model.predict_proba(X) directly; this routes that through our
    module.predict_proba(model, X) contract so it works regardless of what
    fit() returned."""

    def __init__(self, model, module):
        self._model, self._module = model, module

    def predict_proba(self, X):
        return self._module.predict_proba(self._model, X)


class _XPERSubsetAdapter:
    """Like _XPERModelAdapter, but only top_features are exposed to XPER's coalition sampling —
    every other column is held fixed at background_row's real values (a single actual
    observation, so dtypes are guaranteed valid) rather than perturbed. XPER still evaluates
    the real model's real performance; it just isn't asked to attribute it across every
    feature, which is what made runtime tractable at all."""

    def __init__(self, model, module, top_features, background_row):
        self._model, self._module = model, module
        self._top_features = list(top_features)
        self._background_row = background_row

    def predict_proba(self, X):
        n = len(X)
        df = pd.concat([self._background_row] * n, ignore_index=True)
        active = np.asarray(X)
        for i, col in enumerate(self._top_features):
            df[col] = active[:, i]
        df = df.astype(self._background_row.dtypes)  # numpy round-trip drops dtypes — same fix as SHAP's adapter
        return self._module.predict_proba(self._model, df[cm.FEATURES])


def compute_xper(model, module, X_train, y_train, X_test, y_test, metric="AUC", cfp=None, cfn=None, top_features=None):
    if not HAS_XPER:
        return None
    if top_features is not None:
        adapter = _XPERSubsetAdapter(model, module, top_features, X_train.iloc[[0]])
        X_train, X_test = X_train[top_features], X_test[top_features]
    else:
        adapter = _XPERModelAdapter(model, module)
    xper = ModelPerformance(X_train, y_train, X_test, y_test, adapter)
    performance = xper.evaluate([metric], CFP=cfp, CFN=cfn)
    phi, phi_i_j = xper.calculate_XPER_values([metric], CFP=cfp, CFN=cfn, kernel=True)
    return {"performance": performance, "phi": phi, "phi_i_j": phi_i_j, "features": list(top_features) if top_features is not None else list(X_train.columns)}

## Permutation importance — cheap cross-check

In [ ]:
def permutation_importance_report(model, module, X, y, n_repeats=10):
    return cm.manual_permutation_importance(model, module, X, y, n_repeats=n_repeats)

## Global surrogate tree — fidelity-checked approximation

A shallow tree fit to reproduce the model's own predicted probabilities, not the true
labels. Always report fidelity (R² against the real model's output) alongside it — a
surrogate is only useful if you know how far it actually is from the model it's standing in for.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import r2_score

def encode_for_tree(X):
    Xe = X.copy()
    for col in Xe.columns:
        if not pd.api.types.is_numeric_dtype(Xe[col]):
            Xe[col] = Xe[col].astype("category").cat.codes
    return Xe.fillna(-1)


def global_surrogate(model, module, X, max_depth=4):
    probs = module.predict_proba(model, X)[:, 1]
    Xe = encode_for_tree(X)
    surrogate = DecisionTreeRegressor(max_depth=max_depth, random_state=42).fit(Xe, probs)
    fidelity = r2_score(probs, surrogate.predict(Xe))
    return surrogate, fidelity, Xe.columns


def run_surrogate(model, module, X_sample, model_name):
    surrogate, fidelity, cols = global_surrogate(model, module, X_sample)
    print(f"    surrogate fidelity (R² vs. the real model) = {fidelity:.3f}")
    fig, ax = plt.subplots(figsize=(14, 6))
    plot_tree(surrogate, feature_names=list(cols), max_depth=3, filled=True, fontsize=7, ax=ax)
    ax.set_title(f"{model_name} — global surrogate (depth-limited to 3; fidelity R²={fidelity:.3f})")
    plt.tight_layout()
    fig.savefig(NB_OUTPUT_DIR / f"interpretability_surrogate_{model_name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    return fidelity

## PDP + ICE — shape of the relationship, not just attribution

SHAP says what matters; this says what the relationship looks like — monotonic, a hard
threshold, does it vary a lot person-to-person (that's what ICE adds over the PDP average).
Scoped to the same three features used for FPDP (`debt_to_income_ratio`,
`combined_loan_to_value_ratio`, `income`) instead of sweeping all of them.

In [ ]:
PDP_FEATURES = [f for f in ["debt_to_income_ratio", "combined_loan_to_value_ratio", "income"] if f in cm.FEATURES]

def compute_pdp_ice(model, module, X, feature, n_points=20, ice_sample_size=50, random_state=42):
    grid = np.linspace(X[feature].quantile(0.05), X[feature].quantile(0.95), n_points)
    rng = np.random.RandomState(random_state)
    ice_idx = rng.choice(len(X), size=min(ice_sample_size, len(X)), replace=False)
    X_ice = X.iloc[ice_idx].reset_index(drop=True)
    ice_curves = np.zeros((len(X_ice), n_points))
    for j, val in enumerate(grid):
        X_mod = X_ice.copy()
        X_mod[feature] = val
        ice_curves[:, j] = module.predict_proba(model, X_mod)[:, 1]
    return grid, ice_curves.mean(axis=0), ice_curves


def plot_pdp_ice(grid, pdp_curve, ice_curves, feature, model_name):
    fig, ax = plt.subplots(figsize=(6, 4))
    for row in ice_curves:
        ax.plot(grid, row, color="steelblue", alpha=0.1, linewidth=0.8)
    ax.plot(grid, pdp_curve, color="black", linewidth=2, label="PDP (average)")
    ax.set_xlabel(feature)
    ax.set_ylabel("P(approved)")
    ax.set_title(f"{model_name} — PDP + ICE — {feature}")
    ax.legend()
    plt.tight_layout()
    fig.savefig(NB_OUTPUT_DIR / f"interpretability_pdp_ice_{model_name}_{feature}.png", dpi=150, bbox_inches="tight")
    plt.show()


def run_pdp_ice_feature(model, module, X_sample, feature, model_name):
    grid, pdp_curve, ice_curves = compute_pdp_ice(model, module, X_sample, feature)
    plot_pdp_ice(grid, pdp_curve, ice_curves, feature, model_name)
    return {"grid": grid, "pdp_curve": pdp_curve}

## LIME — on the same representative applicants as SHAP

LIME perturbs every feature it's given with Gaussian noise, which breaks HMDA's
integer-coded categoricals (loan type, lien status, etc.) — a perturbed "2.7" isn't a valid
category. So it only perturbs the genuinely continuous features (income, loan amount, DTI,
LTV, property value, loan term) and holds everything else at the instance's real value.
Run on the same instances as SHAP so the two can be compared directly — this is the
"Disagreement in XAI" check from the syllabus (Krishna et al. 2025), not a SHAP replacement.

In [ ]:
import lime.lime_tabular

LIME_CONTINUOUS_COLS = [f for f in [
    "income", "loan_amount", "debt_to_income_ratio", "combined_loan_to_value_ratio",
    "property_value", "loan_term", "intro_rate_period", "total_units",
] if f in cm.FEATURES]
LIME_OTHER_COLS = [f for f in cm.FEATURES if f not in LIME_CONTINUOUS_COLS]


def get_lime_explainer(X_train):
    Xc = X_train[LIME_CONTINUOUS_COLS].fillna(X_train[LIME_CONTINUOUS_COLS].median())
    explainer = lime.lime_tabular.LimeTabularExplainer(
        Xc.values, feature_names=LIME_CONTINUOUS_COLS, class_names=["denied", "approved"],
        mode="classification", random_state=42,
    )
    return explainer, Xc.median()


def lime_explain(explainer, model, module, instance_row, medians, num_features=8):
    def predict_fn(arr):
        df = pd.DataFrame(arr, columns=LIME_CONTINUOUS_COLS)
        for c in LIME_OTHER_COLS:
            df[c] = instance_row[c]
        return module.predict_proba(model, df[cm.FEATURES])
    row_values = instance_row[LIME_CONTINUOUS_COLS].fillna(medians).values
    return explainer.explain_instance(row_values, predict_fn, num_features=num_features)


def pick_representative_instances(model, module, X):
    probs = module.predict_proba(model, X)[:, 1]
    return {
        "approved (highest score)": int(np.argmax(probs)),
        "denied (lowest score)": int(np.argmin(probs)),
        "borderline (closest to threshold)": int(np.argmin(np.abs(probs - cm.TEAM_THRESHOLD))),
    }

## Run across available models

One loop per model. Getting the model is the only hard dependency — if that fails, skip
the model entirely. Every metric after that (SHAP, XPER, permutation importance, surrogate,
each PDP/ICE feature, each LIME instance) goes through `run_step`, so one failing doesn't
take the rest down with it.

`common_metrics.EVAL_SAMPLE_SIZE` caps every row-scaling metric (AUC, XPER, permutation
importance, the surrogate, PDP/ICE) to one subsample drawn once per model and reused across
all of them — not just for speed (XPER's coalition sampling was clocked at ~3 hours on
XGBoost's full 1.37M-row test set before this cap), but so every metric for a given model is
evaluated on the same rows instead of several independent random slices. SHAP's own sample
(200 test rows, 100 background rows for KernelExplainer) stays untouched — that size was
already deliberately small for a different reason (KernelExplainer's own per-row cost).

**Checkpointing**: every `run_step` call gets a `checkpoint_dir` under
`outputs/evaluation/interpretability/checkpoints/<model>/` (or a subfolder of it for the
per-feature/per-instance PDP-ICE and LIME steps) — the result is written to a timestamped file
the moment it's computed, so interrupting mid-run (e.g. XPER taking hours) doesn't lose metrics
that already finished, and re-running loads each metric's latest checkpoint instead of
recomputing it. Set `FORCE_RECOMPUTE = True` to ignore checkpoints and redo everything.

In [ ]:
def compute_auc(model, module, X_test, y_test):
    probs = module.predict_proba(model, X_test)[:, 1]
    return roc_auc_score(y_test, probs)


def run_shap(model, module, model_type, X_train, X_test):
    explainer, method = get_shap_explainer(model_type, model, module, X_train)
    test_sample = X_test.sample(min(200, len(X_test)), random_state=42)
    shap_values = explainer(test_sample) if model_type != "kernel" else explainer.shap_values(test_sample)
    print(f"    SHAP method: {method}")
    return shap_values


def run_lime_instance(model, module, explainer, medians, row, num_features=8):
    return lime_explain(explainer, model, module, row, medians, num_features=num_features)


FORCE_RECOMPUTE = False  # set True to ignore checkpoints and recompute everything
XPER_TOP_N = 5  # restrict XPER to this many features (by permutation importance) — see markdown above

results = {}

for name, module in cm.available_models().items():
    print(f"\n=== {name} ===")
    try:
        model = cm.get_or_fit_model(name, module, X_train, y_train)
    except Exception as e:
        print(f"  model unavailable ({type(e).__name__}: {e}) — skipping {name} entirely")
        continue

    entry = {"model": model}
    model_type = MODEL_EXPLAINER_TYPE[name]
    ckpt_dir = NB_OUTPUT_DIR / "checkpoints" / name

    sample_size = cm.EVAL_SAMPLE_SIZE.get(name)
    X_train_s, y_train_s, X_test_s, y_test_s = X_train, y_train, X_test, y_test
    if sample_size is not None:
        X_train_s = X_train.sample(min(sample_size, len(X_train)), random_state=42)
        y_train_s = y_train.loc[X_train_s.index]
        X_test_s = X_test.sample(min(sample_size, len(X_test)), random_state=42)
        y_test_s = y_test.loc[X_test_s.index]
        print(f"  AUC/XPER/permutation importance/surrogate/PDP-ICE use a {len(X_test_s)}-row subsample for this model")

    if cm.run_step(entry, "auc", compute_auc, model, module, X_test_s, y_test_s, checkpoint_dir=ckpt_dir, force=FORCE_RECOMPUTE):
        print(f"  AUC: {entry['auc']:.4f}")

    cm.run_step(entry, "shap_values", run_shap, model, module, model_type, X_train, X_test, checkpoint_dir=ckpt_dir, force=FORCE_RECOMPUTE)
    cm.run_step(entry, "permutation_importance", permutation_importance_report, model, module, X_test_s, y_test_s, n_repeats=5, checkpoint_dir=ckpt_dir, force=FORCE_RECOMPUTE)

    if "permutation_importance" in entry:
        top_features = sorted(entry["permutation_importance"], key=entry["permutation_importance"].get, reverse=True)[:XPER_TOP_N]
        print(f"  XPER restricted to top {XPER_TOP_N} features by permutation importance: {top_features}")
        cm.run_step(entry, "xper", compute_xper, model, module, X_train_s, y_train_s, X_test_s, y_test_s, metric="AUC", top_features=top_features, checkpoint_dir=ckpt_dir, force=FORCE_RECOMPUTE)
    else:
        print("  xper skipped — no permutation importance ranking to pick features from")

    cm.run_step(entry, "surrogate_fidelity", run_surrogate, model, module, X_test_s, name, checkpoint_dir=ckpt_dir, force=FORCE_RECOMPUTE)

    entry["pdp_ice"] = {}
    for feature in PDP_FEATURES:
        cm.run_step(entry["pdp_ice"], feature, run_pdp_ice_feature, model, module, X_test_s, feature, name, checkpoint_dir=ckpt_dir / "pdp_ice", force=FORCE_RECOMPUTE)

    entry["lime"] = {}
    lime_explainer, lime_medians = get_lime_explainer(X_train)
    for label, idx in pick_representative_instances(model, module, X_test).items():
        row = X_test.iloc[idx]
        if cm.run_step(entry["lime"], label, run_lime_instance, model, module, lime_explainer, lime_medians, row, checkpoint_dir=ckpt_dir / "lime", force=FORCE_RECOMPUTE):
            print(f"    LIME {label}: {entry['lime'][label].as_list()[:2]}")

    results[name] = entry

if not results:
    print("No models ready yet — drop a models/<name>_model.py file in and re-run.")

## Global comparison — mean |SHAP| per feature, across models

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, r in results.items():
        if "shap_values" not in r:
            continue
        sv = r["shap_values"]
        vals = sv.values if hasattr(sv, "values") else np.asarray(sv)
        mean_abs = np.abs(vals).mean(axis=0)
        ax.barh(cm.FEATURES, mean_abs, alpha=0.5, label=name)
    ax.set_xlabel("mean |SHAP value|")
    ax.legend()
    plt.tight_layout()
    fig.savefig(NB_OUTPUT_DIR / "interpretability_shap_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

## Summary table — one row per model, copy-ready for the slide deck

A blank (`—`) means that metric failed for that model, not that the row is missing. Every
model's AUC here is on its own `EVAL_SAMPLE_SIZE` subsample (20,000 rows for xgboost/logreg,
5,000 for TabPFN), not the full test set — a large enough sample for a stable estimate, but
worth noting in the write-up rather than treating it as the full-test-set number.

In [ ]:
def top_xper_feature(entry):
    if "xper" not in entry:
        return "—"
    phi = entry["xper"]["phi"]
    xper_features = entry["xper"]["features"]  # XPER's own feature list, not necessarily all of FEATURES — see XPER_TOP_N
    idx = int(np.argmax(np.abs(phi[1:]))) if len(phi) > 1 else None
    return xper_features[idx] if idx is not None else "—"


def top_shap_feature(entry):
    if "shap_values" not in entry:
        return "—"
    sv = entry["shap_values"]
    vals = sv.values if hasattr(sv, "values") else np.asarray(sv)
    return cm.FEATURES[int(np.argmax(np.abs(vals).mean(axis=0)))]


summary_rows = []
for name, r in results.items():
    summary_rows.append({
        "model": name,
        "AUC": round(r["auc"], 4) if "auc" in r else "—",
        "top SHAP feature": top_shap_feature(r),
        "top XPER feature": top_xper_feature(r),
        "surrogate fidelity (R²)": round(r["surrogate_fidelity"], 3) if "surrogate_fidelity" in r else "—",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

## Smoke test — remove once real models are in `models/`

A throwaway logistic regression on a small sample, just to check the pipeline runs
end to end against the real feature schema. Not a real result.

In [ ]:
if not results:
    print("Running a throwaway smoke test — NOT a real model, just checking the harness works end to end.")
    sample = train_df.sample(20_000, random_state=42)
    Xs, ys = cm.get_X_y(sample)
    smoke_model = cm.SmokeTestModule.fit(Xs, ys)
    smoke_test_sample = X_test.sample(2_000, random_state=42)
    smoke_probs = cm.SmokeTestModule.predict_proba(smoke_model, smoke_test_sample)
    print("Smoke test predict_proba shape:", smoke_probs.shape, "— harness is wired correctly.")

    if HAS_XPER:
        # kernel-based XPER's runtime scales with feature count, not row count — cut to a
        # handful of columns and fit a dedicated small model on just those, so this stays
        # a fast plumbing check (a model fit on all features would reject a 5-column input)
        xper_cols = cm.FEATURES[:5]
        xper_train_sample = Xs[xper_cols]
        xper_smoke_model = cm.SmokeTestModule.fit(xper_train_sample, ys)
        xper_test_sample = X_test.sample(50, random_state=42)[xper_cols]
        xper_smoke = compute_xper(
            xper_smoke_model, cm.SmokeTestModule, xper_train_sample, ys,
            xper_test_sample, y_test.loc[xper_test_sample.index],
            metric="AUC",
        )
        print("XPER smoke test phi shape:", xper_smoke["phi"].shape, "— XPER call succeeded.")

## Save outputs

Writes to `outputs/evaluation/interpretability/` (gitignored, same as `models/` and `data/`)
so results survive independently of the notebook and don't need a re-run to hand to a
teammate or drop into the deck. Individual metrics are already checkpointed as they run (see
above); this writes the final consolidated summary + full results for convenience. The
pickle excludes each entry's `model` object — that's already saved separately as
`models/<name>_model.joblib`, no need to duplicate a 100MB+ artifact inside a results file.

In [ ]:
if results:
    summary_df.to_csv(NB_OUTPUT_DIR / "interpretability_summary.csv", index=False)
    picklable_results = {name: {k: v for k, v in r.items() if k != "model"} for name, r in results.items()}
    with open(NB_OUTPUT_DIR / "interpretability_results.pkl", "wb") as f:
        pickle.dump(picklable_results, f)
    print(f"Saved summary + full results to {NB_OUTPUT_DIR}/")